# 🚀 Stack 2.9 - Kaggle Training

**Free GPU training on Kaggle**

This notebook trains a LoRA adapter for Stack 2.9 on **Qwen2.5-Coder-7B** using Kaggle's free GPU.

⏱️ **Expected runtime:** 2-4 hours
💾 **VRAM needed:** ~16GB (Kaggle P100 has 16GB)

---

**Instructions:**
1. Enable GPU: Settings → Accelerator → GPU T4
2. Run cells in order from the top
3. Model auto-downloads if not present

---

In [ ]:
# STEP 1: Check GPU
import subprocess
subprocess.run(["nvidia-smi"], check=True)
print("✅ GPU ready!")

In [ ]:
# STEP 2: Clone repo and setup paths
import os
import shutil
import subprocess

os.chdir("/kaggle/working")

REPO_DIR = "/kaggle/working/stack-2.9"
MODEL_DIR = os.path.join(REPO_DIR, "base_model_qwen7b")
OUTPUT_DIR = os.path.join(REPO_DIR, "training_output")

# Remove old repo if exists
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

# Clone fresh
subprocess.run(["git", "clone", "https://github.com/my-ai-stack/stack-2.9.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)

print(f"✅ Working in: {os.getcwd()}")

In [ ]:
# STEP 3: Install dependencies
import subprocess

subprocess.run(["pip", "install", "-q", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)
subprocess.run(["pip", "install", "-q", "transformers==4.40.0", "peft==0.10.0", "accelerate==0.34.0", "datasets", "pyyaml", "tqdm", "scipy", "bitsandbytes==0.43.0"], check=True)
print("✅ Dependencies installed")

In [ ]:
# STEP 4: Prepare training data
import os
import json

# Check for available training data
REPO_TRAIN_DATA = os.path.join(REPO_DIR, "training-data/final/train.jsonl")
MINI_DATA_DIR = os.path.join(REPO_DIR, "data_mini")
MINI_DATA_FILE = os.path.join(MINI_DATA_DIR, "train_mini.jsonl")
SYNTHETIC_DATA_FILE = os.path.join(REPO_DIR, "data/synthetic.jsonl")

print("🔍 Checking for training data...")

if os.path.exists(REPO_TRAIN_DATA):
    print(f"   Found full dataset: {REPO_TRAIN_DATA}")
    os.makedirs(MINI_DATA_DIR, exist_ok=True)
    if not os.path.exists(MINI_DATA_FILE):
        print("   Creating mini dataset (1000 samples)...")
        import subprocess
        subprocess.run(["python", os.path.join(REPO_DIR, "scripts/create_mini_dataset.py"),
                       "--size", "1000", "--output", MINI_DATA_FILE, "--source", REPO_TRAIN_DATA], check=True)
    DATA_FILE = MINI_DATA_FILE
    
elif os.path.exists(MINI_DATA_FILE):
    DATA_FILE = MINI_DATA_FILE
    print(f"   Using existing mini dataset: {MINI_DATA_FILE}")

else:
    print("   No dataset found. Creating synthetic data...")
    
    # Simple code completion examples
    examples = [
        {"instruction": "Write a Python function to reverse a string", 
         "output": "def reverse_string(s):\n    return s[::-1]"},
        {"instruction": "Write a function to check if a number is prime", 
         "output": "def is_prime(n):\n    if n <= 1:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True"},
        {"instruction": "Write a binary search function", 
         "output": "def binary_search(arr, target):\n    left, right = 0, len(arr) - 1\n    while left <= right:\n        mid = (left + right) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            left = mid + 1\n        else:\n            right = mid - 1\n    return -1"},
    ]
    
    samples = []
    for i in range(1000):
        for ex in examples:
            samples.append(ex)
    
    os.makedirs(os.path.dirname(SYNTHETIC_DATA_FILE), exist_ok=True)
    with open(SYNTHETIC_DATA_FILE, 'w') as f:
        for s in samples:
            f.write(json.dumps(s) + '\n')
    
    DATA_FILE = SYNTHETIC_DATA_FILE
    print(f"   Created synthetic dataset: {len(samples)} samples")

print(f"\n✅ Using training data: {DATA_FILE}")
print(f"   Size: {os.path.getsize(DATA_FILE) / 1024:.1f} KB")

In [ ]:
# STEP 5: Prepare config for training
import yaml
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

config = {
    'model': {
        'name': 'Qwen/Qwen2.5-Coder-7B',
        'trust_remote_code': True,
        'torch_dtype': 'float16'
    },
    'data': {
        'input_path': DATA_FILE,
        'max_length': 2048,
        'train_split': 1.0
    },
    'lora': {
        'r': 16,
        'alpha': 32,
        'dropout': 0.05,
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        'bias': 'none',
        'task_type': 'CAUSAL_LM'
    },
    'training': {
        'num_epochs': 1,
        'batch_size': 2,
        'gradient_accumulation': 4,
        'learning_rate': 2e-4,
        'warmup_steps': 50,
        'weight_decay': 0.01,
        'max_grad_norm': 1.0,
        'logging_steps': 10,
        'save_steps': 100,
        'save_total_limit': 2,
        'fp16': True,
        'bf16': False,
        'gradient_checkpointing': True
    },
    'output': {
        'lora_dir': os.path.join(OUTPUT_DIR, 'lora'),
        'logging_dir': os.path.join(OUTPUT_DIR, 'logs')
    },
    'quantization': {'enabled': False},
    'hardware': {'device': 'cuda', 'num_gpus': 1, 'use_4bit': False, 'use_8bit': False}
}

config_path = os.path.join(OUTPUT_DIR, "train_config.yaml")
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Config saved: {config_path}")
print(f"   Model: {config['model']['name']}")
print(f"   Data: {config['data']['input_path']}")

In [ ]:
# STEP 6: Train LoRA
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "stack_2_9_training"))

print("="*60)
print("STARTING TRAINING")
print("="*60)

from stack_2_9_training.train_lora import train_lora

try:
    trainer = train_lora(config_path)
    print("\n" + "="*60)
    print("TRAINING COMPLETED")
    print("="*60)
except Exception as e:
    print(f"\n❌ Training failed: {e}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# STEP 7: Merge LoRA adapter
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "stack_2_9_training"))
from stack_2_9_training.merge_adapter import merge_adapter

lora_dir = config['output']['lora_dir']
merged_dir = os.path.join(OUTPUT_DIR, 'merged')
os.makedirs(merged_dir, exist_ok=True)

print("="*60)
print("MERGING")
print("="*60)

try:
    merge_adapter(
        base_model_name_or_path=config['model']['name'],
        adapter_path=lora_dir,
        output_path=merged_dir,
        use_safetensors=True
    )
    print("\n✅ Merge completed!")
    print(f"Files: {os.listdir(merged_dir)}")
except Exception as e:
    print(f"\n❌ Merge failed: {e}")
    import traceback
    traceback.print_exc()
    raise

print("\n" + "="*60)
print("🎉 ALL DONE!")
print("="*60)
print(f"\n📦 Model ready: {merged_dir}")
print("\n⏳ Download 'merged' folder from Kaggle Output panel before session ends!")